# Smart MCQ Solver
The goal of this competition is to build intelligent models that can accurately predict the top three most probable answers for challenging multiple choice questions. Participants are encouraged to develop efficient AI and machine learning solutions capable of strong reasoning and answer ranking.

# *Exploratory Data Analysis [EDA]*

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
# Dataset Dimensions
print(f"Train rows: {train_df.shape[0]}, Columns: {train_df.shape[1]}")
print(f"Test rows:  {test_df.shape[0]}, Columns: {test_df.shape[1]}")

In [ ]:
# null Values
train_df.isnull().sum()

In [ ]:
# Class distribution
print(train_df['answer'].value_counts().sort_index())

plt.figure(figsize=(6, 4))
sns.countplot(x='answer', data=train_df, order=['A', 'B', 'C', 'D', 'E'], palette='viridis')
plt.title("Distribution of Correct Answers (Train)")
plt.xlabel("Answer Choice")
plt.ylabel("Count")
plt.show()

In [ ]:
# Text Lengths 
train_df['prompt_word_count'] = train_df['prompt'].apply(lambda x: len(str(x).split()))
test_df['prompt_word_count'] = test_df['prompt'].apply(lambda x: len(str(x).split()))

options_cols = ['A', 'B', 'C', 'D', 'E']
for col in options_cols:
    train_df[f'{col}_len'] = train_df[col].apply(lambda x: len(str(x).split()))

train_df['avg_option_word_count'] = train_df[[f'{col}_len' for col in options_cols]].mean(axis=1)


print(train_df[['prompt_word_count', 'avg_option_word_count']].describe())

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(train_df['prompt_word_count'], kde=True, color='blue', label='Train Prompts', alpha=0.6)
sns.histplot(test_df['prompt_word_count'], kde=True, color='orange', label='Test Prompts', alpha=0.6)
plt.axvline(train_df['prompt_word_count'].max(), color='red', linestyle='--', label=f"Max Train Words: {train_df['prompt_word_count'].max()}")
plt.title("Prompt Word Count Distribution Comparison")
plt.xlabel("Number of Words")
plt.ylabel("Density")
plt.legend()
plt.show()

### *Summary of EDA*

**Class Distribution Balance:** *There is no heavy class imbalance, a naive static "Majority Class" baseline yields an expected* **mAP@3 score of ~0.20 to 0.25**. *This establishes our Baseline.*

**Prompt Word Lengths:** *The distribution shows that the majority of question prompts are concise, with a 75th percentile sitting around 35 to 45 words, and the absolute maximum rarely exceeding 80 words.*

**Option Word Lengths:** *Answer choices (Options A through E) are highly compact, averaging between 5 to 15 words per choice.*

**Optimizing the Context Window (`max_length`):** *When tokenizing text for our deep learning models (PyTorch text vectors and DistilBERT), we must define a hard maximum sequence length. Based on this outcome, setting a configuration is possible.*

# *Text Preprocessing & Baseline*

### *Text Cleaning*
*Removing NaN values, normalize the text to lowercase, remove punctuation, and tokenize it.*
*Help's in model traing*

In [ ]:
columns_to_clean = ['prompt', 'A', 'B', 'C', 'D', 'E']
punctuations = [".", ",", "?", "!", "\"", "'", ":", ";", "(", ")", "-"]

for df in [train_df, test_df]:
    for col in columns_to_clean:
        df[col] = df[col].fillna("")
        df[col] = df[col].str.lower()
    
        for p in punctuations:
            df[col] = df[col].str.replace(p, "", regex=False)

print(train_df['prompt'].iloc[0])

### *Wandb Setup*

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
wb_key = user_secrets.get_secret("wanb-key")

In [ ]:
!pip install wandb

In [ ]:
import wandb
wandb.login(key=wb_key)
wandb.init(project="dl-genai-project", name="tfidf-cosine-baseline")

### *TF-IDF, Cosine Similarity*
*Converting text into numbers using TF-IDF, comparing the prompt with the options using Cosine Similarity, ranks the top 3 options, calculating our performance using mAP@3*

In [ ]:
# converting train_df in 1D for TfidfVectorizer 
all_text = train_df[['prompt', 'A', 'B', 'C', 'D', 'E']].astype(str).values.flatten()

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
vectorizer.fit(all_text)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

predictions = []
options = ['A', 'B', 'C', 'D', 'E']

for index, row in train_df.iterrows():
   
    prompt_text = [row['prompt']]
    options_text = [row['A'], row['B'], row['C'], row['D'], row['E']]
    
    prompt_vector = vectorizer.transform(prompt_text)
    options_vectors = vectorizer.transform(options_text)
    
    similarity_scores = cosine_similarity(prompt_vector, options_vectors)[0]

    # Matching each score to its corresponding option 
    score_pairs = []
    for i in range(5):
        pair = (similarity_scores[i], options[i])
        score_pairs.append(pair)

    score_pairs.sort(reverse=True)
    
    top_3 = [score_pairs[0][1], score_pairs[1][1], score_pairs[2][1]]
    predictions.append(top_3)

train_df['tfidf_prediction'] = predictions

In [ ]:
def calculate_map3(actuals, predictions):
    
    total_score = 0.0
    num_questions = len(actuals)
    
    for actual, top_3 in zip(actuals, predictions):
        if actual == top_3[0]:
            total_score += 1.0
        elif actual == top_3[1]:
            total_score += 0.5
        elif actual == top_3[2]:
            total_score += 1.0 / 3.0 
            
    return total_score / num_questions

In [ ]:
map3_score = calculate_map3(train_df['answer'], train_df['tfidf_prediction'])
print("Baseline mAP@3 Score:", map3_score)
wandb.log({"baseline_map3": map3_score})

wandb.finish()

#### Metric Evaluation 
*  Mean Average Precision @ 3 ($mAP@3$) evaluates the model’s ability to rank the correct choice within its top three predictions : 
    *   Correct answer ranked at **1st Guess** = $1.0$ point
    *   Correct answer ranked at **2nd Guess** = $0.5$ point
    *   Correct answer ranked at **3rd Guess** = $0.333$ point
    *   Correct answer ranked **beyond 3rd or missed** = $0.0$ points
*   **The Baseline Score:** Averaging these individual precision values across all dataset rows yields our static baseline evaluation metric ($~0.2576$ $mAP@3$).

# *Neural Network Model [Scratch Model]*


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import wandb

wandb.init(project="dl-genai-project", name="NN-model")

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer_scratch = TfidfVectorizer(stop_words='english', max_features=1000)

In [ ]:
all_text_scratch = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    all_text_scratch.extend(train_df[col].tolist())
vectorizer_scratch.fit(all_text_scratch)

## *Vector Concatenation*
*A neural network needs to look at the question and an answer together to decide if they make a good pair.*

In [ ]:
X_prompt = vectorizer_scratch.transform(train_df['prompt']).toarray()
X_A = vectorizer_scratch.transform(train_df['A']).toarray()
X_B = vectorizer_scratch.transform(train_df['B']).toarray()
X_C = vectorizer_scratch.transform(train_df['C']).toarray()
X_D = vectorizer_scratch.transform(train_df['D']).toarray()
X_E = vectorizer_scratch.transform(train_df['E']).toarray()

In [ ]:
num_rows = len(train_df)
X_features = np.zeros((num_rows, 5, 2000))  # 5 options, each has size (1000 prompt + 1000 option)

for i in range(num_rows):
    X_features[i, 0] = np.concatenate([X_prompt[i], X_A[i]])
    X_features[i, 1] = np.concatenate([X_prompt[i], X_B[i]])
    X_features[i, 2] = np.concatenate([X_prompt[i], X_C[i]])
    X_features[i, 3] = np.concatenate([X_prompt[i], X_D[i]])
    X_features[i, 4] = np.concatenate([X_prompt[i], X_E[i]])

label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
y_labels = train_df['answer'].map(label_map).values

In [ ]:
X_tensor = torch.tensor(X_features, dtype=torch.float32)
y_tensor = torch.tensor(y_labels, dtype=torch.long)

*builting a PyTorch model with a single nn.Linear(2000, 1) layer. During the forward pass, you flattened the batch, squashed the 2000 features into 1 single "confidence score," and then reshaped it back to show 5 scores per question.*

In [ ]:
class SimpleMCQNet(nn.Module):
    def __init__(self):
        super(SimpleMCQNet, self).__init__()
        self.linear = nn.Linear(2000, 1)
        
    def forward(self, x):
        batch_size = x.size(0)
        x_flat = x.view(batch_size * 5, -1)
        scores_flat = self.linear(x_flat)
        return scores_flat.view(batch_size, 5)

model = SimpleMCQNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

In [ ]:
# training loop
print("Training the custom scratch model...")
for epoch in range(5):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)
    
    # Backward pass 
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}/5 - Loss: {loss.item():.4f}")
    wandb.log({"epoch": epoch+1, "train_loss": loss.item()})

In [ ]:
model.eval()
with torch.no_grad():
    raw_predictions = model(X_tensor).numpy()

letters = ['A', 'B', 'C', 'D', 'E']
all_top_3_predictions = []

for i in range(num_rows):
    ranked_indices = np.argsort(raw_predictions[i])[::-1]
    top_3_letters = [letters[idx] for idx in ranked_indices[:3]]
    all_top_3_predictions.append(top_3_letters)

scratch_map3 = calculate_map3(train_df['answer'], all_top_3_predictions)
print(f"Scratch mAP@3 Score is: {scratch_map3:.4f}")

wandb.log({"scratch_model_map3": scratch_map3})
wandb.finish()

## *Test Split Inference & Submission Generation*

In [ ]:
X_prompt_test = vectorizer_scratch.transform(test_df['prompt']).toarray()
X_A_test = vectorizer_scratch.transform(test_df['A']).toarray()
X_B_test = vectorizer_scratch.transform(test_df['B']).toarray()
X_C_test = vectorizer_scratch.transform(test_df['C']).toarray()
X_D_test = vectorizer_scratch.transform(test_df['D']).toarray()
X_E_test = vectorizer_scratch.transform(test_df['E']).toarray()

num_test_rows = len(test_df)
X_features_test = np.zeros((num_test_rows, 5, 2000))

for i in range(num_test_rows):
    X_features_test[i, 0] = np.concatenate([X_prompt_test[i], X_A_test[i]])
    X_features_test[i, 1] = np.concatenate([X_prompt_test[i], X_B_test[i]])
    X_features_test[i, 2] = np.concatenate([X_prompt_test[i], X_C_test[i]])
    X_features_test[i, 3] = np.concatenate([X_prompt_test[i], X_D_test[i]])
    X_features_test[i, 4] = np.concatenate([X_prompt_test[i], X_E_test[i]])

X_tensor_test = torch.tensor(X_features_test, dtype=torch.float32)

In [ ]:
# model.eval()
# with torch.no_grad():
#     test_outputs = model(X_tensor_test).numpy()

# letters = ['A', 'B', 'C', 'D', 'E']
# test_predictions = []

# for i in range(num_test_rows):
#     ranked_indices = np.argsort(test_outputs[i])[::-1]
#     top_3_letters = [letters[idx] for idx in ranked_indices[:3]]
    
#     prediction_string = " ".join(top_3_letters)
#     test_predictions.append(prediction_string)

# scrach_submission = pd.DataFrame({
#     'id': test_df['id'],
#     'prediction': test_predictions
# })

# scrach_submission.to_csv("scrach_submission.csv", index=False)
# print("Submission file successfully created as 'submission.csv'!")

# print(scrach_submission.head())

# *Enter the Transformers*
## *Pretrained Model*

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, pipeline
from datasets import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
hg_dataset = Dataset.from_pandas(train_df)

## *BERT/RoBERTa Architecture & Attention Mechanisms*

*BERT (Bidirectional Encoder Representations from Transformers) and its optimized successor, RoBERTa, rely on the Transformer Encoder architecture. Unlike older models that read text sequentially, transformers read the entire sequence at once.*

*The core innovation is the Self-Attention Mechanism. It allows the model to weigh the importance of every word in a sentence relative to every other word, dynamically capturing context.*

### *Generating Context-Aware Embeddings*
*Traditional embeddings (like Word2Vec) assign a single, static vector to a word regardless of its context. Pre-trained transformer models generate context-aware embeddings*


## *Zero-Shot Classification*
*Zero shot classification allows a model to categorize text into classes it has never explicitly been trained on. By leveraging a model trained on Natural Language Inference (NLI) tasks where it learns to determine if a premise entails a hypothesis we can pass our Kaggle question as the premise and our A-E options as hypotheses. The model calculates the probability that the prompt logically leads to each option.*

In [ ]:
zero_shot_clf = pipeline(
    "zero-shot-classification", 
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

sample_row = train_df.iloc[0]
prompt = sample_row['prompt']
candidate_options = [
    sample_row['A'], 
    sample_row['B'], 
    sample_row['C'], 
    sample_row['D'], 
    sample_row['E']
]

result = zero_shot_clf(prompt, candidate_labels=candidate_options)

In [ ]:
# from transformers import pipeline

# wandb.init(project="dl-genai-project", name="zero-shot-mnli-baseline", reinit=True)

# submission_results = []
# options_cols = ['A', 'B', 'C', 'D', 'E']

# for _, row in test_df.iterrows():
#     prompt = row['prompt']
#     options_map = {col: row[col] for col in options_cols}
#     candidate_options = list(options_map.values())
    
#     result = zero_shot_clf(prompt, candidate_labels=candidate_options)
#     ranked_labels = result['labels']
    
#     reverse_map = {val: key for key, val in options_map.items()}
#     top_3_letters = [reverse_map[text] for text in ranked_labels[:3]]
    
#     prediction_string = " ".join(top_3_letters)
#     submission_results.append({'id': row['id'], 'Prediction': prediction_string})

# zero_submission = pd.DataFrame(submission_results)
# zero_submission.to_csv('submission_zeroshot.csv', index=False)

# wandb.finish()

## *Data Tokenization & Formatting*
*Initializing the DistilBERT tokenizer and formatting the prompt-option pairs into Hugging Face Dataset structures for transformer model training.*


In [ ]:
model_name = "distilbert-base-uncased"
fine_tune_model = AutoModelForMultipleChoice.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

def format_mcq_data(dataframe, is_test=False):
    input_ids_list = []
    attention_mask_list = []
    label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
    
    for _, row in dataframe.iterrows():
        pairs = [[row['prompt'], row[opt]] for opt in ['A', 'B', 'C', 'D', 'E']]
        
        encoded_pairs = tokenizer(
            pairs,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )
        
        input_ids_list.append(encoded_pairs['input_ids'].numpy())
        attention_mask_list.append(encoded_pairs['attention_mask'].numpy())
            
    formatted_dict = {
        'input_ids': input_ids_list,
        'attention_mask': attention_mask_list
    }
    
    if not is_test:
        formatted_dict['label'] = dataframe['answer'].map(label_map).tolist()
        
    return Dataset.from_dict(formatted_dict)

hf_train_dataset = format_mcq_data(train_df, is_test=False)
hf_test_dataset = format_mcq_data(test_df, is_test=True)

print(hf_train_dataset)

## Fine-Tuning the Transformer Model
*Initializing the DistilBERT model for multiple choice, defining the training arguments to optimize GPU usage, and executing the fine-tuning loop. Model performance is evaluated using our custom mAP@3 metric and logged directly to Weights & Biases.*

## Fine-Tuning DistilBERT for Multiple Choice
*Formatting the dataset into prompt-option pairs, initializing the pre-trained transformer, and executing the fine-tuning loop. Model predictions are then evaluated using our custom mAP@3 metric and logged to Weights & Biases.*

In [ ]:
wandb.init(project="dl-genai-project", name="distilbert-finetuned-mcq")

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=10,            
    weight_decay=0.01,            
    logging_steps=10,
    report_to="wandb", 
    save_strategy="no"
)

trainer = Trainer(
    model=fine_tune_model,
    args=training_args,
    train_dataset=hf_train_dataset
)

trainer.train()

raw_predictions = trainer.predict(hf_train_dataset)
logits = raw_predictions.predictions 

letters = ['A', 'B', 'C', 'D', 'E']
all_top_3_predictions = [
    [letters[idx] for idx in np.argsort(logits[i])[::-1][:3]]
    for i in range(len(train_df))
]

finetuned_map3 = calculate_map3(train_df['answer'].tolist(), all_top_3_predictions)

print(f"\nFine-Tuned Transformer mAP@3 Score: {finetuned_map3}")

wandb.log({"finetuned_transformer_map3": finetuned_map3})
wandb.finish()


In [ ]:
wandb.init(project="dl-genai-project", name="distilbert-finetuned-mcq")

## Submission Generation
*Using the fine-tuned DistilBERT model to predict answers for the test dataset, extracting the top 3 ranked choices per question, and formatting them into the final CSV required for the Kaggle competition.*

In [ ]:
test_predictions_raw = trainer.predict(hf_test_dataset)
test_logits = test_predictions_raw.predictions

letters = ['A', 'B', 'C', 'D', 'E']
test_predictions_strings = [
    " ".join([letters[idx] for idx in np.argsort(test_logits[i])[::-1][:3]])
    for i in range(len(test_df))
]

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': test_predictions_strings
})

submission_df.to_csv("submission.csv", index=False)
print(submission_df.head())

# *Retrieval-Augmented Generation[RAG]*
*Retrieval-Augmented Generation (RAG) acts like an open-book test. Instead of forcing the model to rely purely on its internal memory, we use a Vector Database to search for the relevant facts, retrieve a specific "Context Snippet," and feed that snippet directly into the prompt before the model reads the multiple-choice options.*

## *Vector Database Initialization*
*General LLMs often hallucinate or lack domain-specific knowledge for complex science or philosophy MCQs. To fix this, we build a Retrieval-Augmented Generation (RAG) pipeline.*

*FAISS index, is the industry standard for fast similarity search.*

# *Weighted Ensemble*
*We calculate the TF-IDF cosine similarities for the test dataset. Then, we use the softmax function to convert both the raw DeBERTa logits and the TF-IDF scores into 0-to-1 probability distributions. By applying a weighted sum (e.g., 85% DeBERTa, 15% TF-IDF), we get a blended prediction that catches errors either model would make individually.*